In [20]:
import numpy as np
import pandas as pd
from pathlib import Path

In [21]:
!ls ../raw/unsw_nb15/data

NetFlow_v3_Features.csv
NF-UNSW-NB15-v3.csv


In [22]:
root = Path('../raw/unsw_nb15/data')
features = pd.read_csv(root /'NetFlow_v3_Features.csv')
flows = pd.read_csv(root / 'NF-UNSW-NB15-v3.csv')

In [23]:
features

,Feature,Description
0,IPV4_SRC_ADDR,IPv4 source address
1,IPV4_DST_ADDR,IPv4 destination address
2,L4_SRC_PORT,IPv4 source port number
3,L4_DST_PORT,IPv4 destination port number
4,PROTOCOL,IP protocol identifier byte
5,L7_PROTO,Layer 7 protocol (numeric)
6,IN_BYTES,Incoming number of bytes
7,OUT_BYTES,Outgoing number of bytes
8,IN_PKTS,Incoming number of packets
9,OUT_PKTS,Outgoing number of packets


In [80]:
non_ordinal = [
    'IPV4_SRC_ADDR',
    'IPV4_DST_ADDR',
    'L4_SRC_PORT',
    'L4_DST_PORT',
    'PROTOCOL',
    'L7_PROTO',
    'ICMP_TYPE',
    'ICMP_IPV4_TYPE',
    'DNS_QUERY_ID',
    'DNS_QUERY_TYPE',
    'DNS_TTL_ANSWER',
    'FTP_COMMAND_RET_CODE',
]

for col in non_ordinal:
    print(
        col, len(pd.unique(flows[col])))

IPV4_SRC_ADDR 38
IPV4_DST_ADDR 38
L4_SRC_PORT 64568
L4_DST_PORT 64588
PROTOCOL 10
L7_PROTO 125
ICMP_TYPE 1187
ICMP_IPV4_TYPE 256
DNS_QUERY_ID 63678
DNS_QUERY_TYPE 9
DNS_TTL_ANSWER 92
FTP_COMMAND_RET_CODE 16


In [81]:
high_count_categorical = [
    'L4_SRC_PORT', 'L4_DST_PORT', 'ICMP_TYPE', 'DNS_QUERY_ID',
    'PROTOCOL', 'L7_PROTO', 'DNS_TTL_ANSWER', 'ICMP_IPV4_TYPE'
]

for c in high_count_categorical:
    non_ordinal.remove(c)
low_count_categorical = non_ordinal

targets = ('Label', 'Attack')

numerical = [c for c in flows.columns 
             if c not in low_count_categorical 
             and c not in high_count_categorical
             and c not in targets]


print(f'numerical: {numerical}')
print(f'categorical: {low_count_categorical}')
print(f'non sparse categorical: {high_count_categorical}')

numerical: ['FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS', 'IN_BYTES', 'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS', 'TCP_FLAGS', 'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS', 'DURATION_IN', 'DURATION_OUT', 'MIN_TTL', 'MAX_TTL', 'LONGEST_FLOW_PKT', 'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN', 'SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES', 'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS', 'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS', 'SRC_TO_DST_AVG_THROUGHPUT', 'DST_TO_SRC_AVG_THROUGHPUT', 'NUM_PKTS_UP_TO_128_BYTES', 'NUM_PKTS_128_TO_256_BYTES', 'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES', 'NUM_PKTS_1024_TO_1514_BYTES', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT', 'SRC_TO_DST_IAT_MIN', 'SRC_TO_DST_IAT_MAX', 'SRC_TO_DST_IAT_AVG', 'SRC_TO_DST_IAT_STDDEV', 'DST_TO_SRC_IAT_MIN', 'DST_TO_SRC_IAT_MAX', 'DST_TO_SRC_IAT_AVG', 'DST_TO_SRC_IAT_STDDEV']
categorical: ['IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'DNS_QUERY_TYPE', 'FTP_COMMAND_RET_CODE']
non

In [82]:
# filter inf values
print(len(flows))
for c in flows.columns:
    flows = flows[flows[c] != np.inf]
print(len(flows))

2242931
2242931


In [89]:
flows = flows.sort_values(by='FLOW_START_MILLISECONDS')

In [90]:
flows.columns

Index(['FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS', 'IPV4_SRC_ADDR',
       'L4_SRC_PORT', 'IPV4_DST_ADDR', 'L4_DST_PORT', 'PROTOCOL', 'L7_PROTO',
       'IN_BYTES', 'IN_PKTS', 'OUT_BYTES', 'OUT_PKTS', 'TCP_FLAGS',
       'CLIENT_TCP_FLAGS', 'SERVER_TCP_FLAGS', 'FLOW_DURATION_MILLISECONDS',
       'DURATION_IN', 'DURATION_OUT', 'MIN_TTL', 'MAX_TTL', 'LONGEST_FLOW_PKT',
       'SHORTEST_FLOW_PKT', 'MIN_IP_PKT_LEN', 'MAX_IP_PKT_LEN',
       'SRC_TO_DST_SECOND_BYTES', 'DST_TO_SRC_SECOND_BYTES',
       'RETRANSMITTED_IN_BYTES', 'RETRANSMITTED_IN_PKTS',
       'RETRANSMITTED_OUT_BYTES', 'RETRANSMITTED_OUT_PKTS',
       'SRC_TO_DST_AVG_THROUGHPUT', 'DST_TO_SRC_AVG_THROUGHPUT',
       'NUM_PKTS_UP_TO_128_BYTES', 'NUM_PKTS_128_TO_256_BYTES',
       'NUM_PKTS_256_TO_512_BYTES', 'NUM_PKTS_512_TO_1024_BYTES',
       'NUM_PKTS_1024_TO_1514_BYTES', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT',
       'ICMP_TYPE', 'ICMP_IPV4_TYPE', 'DNS_QUERY_ID', 'DNS_QUERY_TYPE',
       'DNS_TTL_ANSWER', 'FTP_COMMAN

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder, StandardScaler, LabelEncoder)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True,
            dtype="float32"
        ), low_count_categorical + ['Attack']),
        # ('lab', LabelEncoder(), non_sparse_categorical),
        ("num", StandardScaler(), numerical),
    ]
)

In [ ]:
from sklearn.preprocessing import (LabelEncoder)

def get_window(processed_flows, size, linegraph=False):
    n_flows = len(processed_flows)
    for start in range(0, n_flows-1, size):
        window_flows = processed_flows.iloc[start:start+size]

        # convert to bidirection flow graph.
        # TODO: port numbers aswell
        def to_graph(flows, edge_ids=('IPV4_SRC_ADDR', 'IPV4_DST_ADDR')):
            ip_addr_le = LabelEncoder()
            edge_list_df = ip_addr_le.fit_transform(flows[edge_ids])
            edge_list_df.columns = ['scr', 'dst']



